# Emailing Participants with SendGrid or SMTP2GO

This notebook is used to email participants with their API keys. This is not to be used by participants themselves, but rather by the workshop organizers to send out keys.

However, you can take a look at the code to see how bulk e-mailing is done using SendGrid or SMTP2GO.

SendGrid (from Twilio) and [SMTP2GO](https://www.smtp2go.com/) are services that allow you to send emails through an API.
You can use either provider to send emails to participants with their API keys.

You can sign up for a free account at [SendGrid](https://sendgrid.com/).

There are other services that allow you to send emails in bulk, such as [Mailgun](https://www.mailgun.com/) and [Amazon SES](https://aws.amazon.com/ses/).

In old days this was done using `smtplib` and `email` libraries, however, these days most e-mail providers have limits on how many emails you can send per day, so it is better to use a service that is designed for this purpose.
This notebook will show you how to use SendGrid or SMTP2GO to send emails to participants with their API keys.

In [ ]:
import os
import requests

# Keep API keys in environment variables and never print the keys themselves.
my_sendgrid_key = os.getenv("SENDGRID_API")
my_smtp2go_key = os.getenv("SMTP2GO_API_KEY")

# SMTP2GO has verified this sender address.
SELF_EMAIL = "valdis.saulespurens@lnb.lv"
DEFAULT_TEST_DESTINATION = "valdis.saulespurens@gmail.com"
sendgrid_sender_email = os.getenv("SENDGRID_FROM_EMAIL")
smtp2go_sender_email = os.getenv("SMTP2GO_FROM_EMAIL", SELF_EMAIL)

if my_sendgrid_key:
    print("SendGrid API key is set.")
if my_smtp2go_key:
    print("SMTP2GO API key is set.")
if not (my_sendgrid_key or my_smtp2go_key):
    print("No email API key found in the environment; pass a key directly when calling email_single().")


## Sending one test email

`email_single(provider, key, message, destination=DEFAULT_TEST_DESTINATION)` sends a plain-text test message to the supplied destination. The destination defaults to `valdis.saulespurens@gmail.com`. Provider names are case-insensitive and may be `SendGrid` or `SMTP2GO`. The function returns a small result dictionary without exposing the API key.

SMTP2GO uses the verified `valdis.saulespurens@lnb.lv` sender address by default. SendGrid continues to use `SENDGRID_FROM_EMAIL`. The participant loop uses SendGrid when its key is available and otherwise uses SMTP2GO; set `EMAIL_PROVIDER` explicitly to override that choice.


In [ ]:
SMTP2GO_SEND_URL = "https://api.smtp2go.com/v3/email/send"
TEST_EMAIL_SUBJECT = "BSSDH email provider test"


def _normalise_provider(provider):
    if not isinstance(provider, str):
        raise TypeError("provider must be a string")

    normalised = provider.strip().upper()
    if normalised not in {"SENDGRID", "SMTP2GO"}:
        raise ValueError("provider must be either 'SendGrid' or 'SMTP2GO'")
    return normalised


def _send_email(provider, key, to_email, subject, message):
    provider = _normalise_provider(provider)
    if not isinstance(key, str) or not key.strip():
        raise ValueError(f"A non-empty {provider} API key is required")
    if not isinstance(message, str) or not message.strip():
        raise ValueError("message must be a non-empty string")

    if provider == "SENDGRID":
        try:
            from sendgrid import SendGridAPIClient
            from sendgrid.helpers.mail import Mail
        except ImportError as exc:
            raise ImportError("SendGrid support requires the sendgrid package") from exc

        if not sendgrid_sender_email:
            raise ValueError("SENDGRID_FROM_EMAIL environment variable not set")
        mail = Mail(
            from_email=sendgrid_sender_email,
            to_emails=to_email,
            subject=subject,
            plain_text_content=message,
        )
        response = SendGridAPIClient(key.strip()).send(mail)
        if not 200 <= response.status_code < 300:
            raise RuntimeError(f"SendGrid request failed with HTTP {response.status_code}")
        return {"provider": "SendGrid", "status_code": response.status_code}

    response = requests.post(
        SMTP2GO_SEND_URL,
        headers={
            "Accept": "application/json",
            "X-Smtp2go-Api-Key": key.strip(),
        },
        json={
            "sender": smtp2go_sender_email,
            "to": [to_email],
            "subject": subject,
            "text_body": message,
        },
        timeout=30,
    )
    try:
        payload = response.json()
    except ValueError:
        payload = {}

    response_data = payload.get("data", {}) if isinstance(payload, dict) else {}
    if not response.ok:
        detail = response_data.get("error") or response.reason
        raise RuntimeError(f"SMTP2GO request failed with HTTP {response.status_code}: {detail}")
    if response_data.get("failed", 0):
        raise RuntimeError(f"SMTP2GO did not accept the email: {response_data.get('failures', [])}")

    return {
        "provider": "SMTP2GO",
        "status_code": response.status_code,
        "email_id": response_data.get("email_id"),
    }


def email_single(provider, key, message, destination=DEFAULT_TEST_DESTINATION):
    if not isinstance(destination, str) or not destination.strip():
        raise ValueError("destination must be a non-empty string")
    destination = destination.strip()

    result = _send_email(
        provider=provider,
        key=key,
        to_email=destination,
        subject=TEST_EMAIL_SUBJECT,
        message=message,
    )
    print(
        f"Test email sent to {destination} with {result['provider']}. "
        f"HTTP status: {result['status_code']}"
    )
    return result


In [ ]:
# This sends a real test email to DEFAULT_TEST_DESTINATION when the cell is run.
email_single(
    provider="SMTP2GO",
    key=my_smtp2go_key,
    message="Hello this is a test",
)


## Reading e-mail participants from XLSX

Our participants are stored in an XLSX file, which we will read using `pandas`. We will then extract the email addresses and API keys from the DataFrame.



In [ ]:
# Load the provisioned participant spreadsheet from a private temp directory.
from pathlib import Path
import pandas as pd
print(f"pandas version: {pd.__version__}")

emails_file = Path("../temp/BSSDH_2025_provisioned_keys.xlsx")
if not emails_file.exists():
    raise FileNotFoundError(f"Emails file {emails_file} does not exist.")

df = pd.read_excel(emails_file, engine="openpyxl")
required_columns = {"E-mail", "Name", "Surname", "api_key"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Missing required participant columns: {sorted(missing_columns)}")

print(f"Loaded participant table with {len(df)} rows and {len(df.columns)} columns.")
print("Expected participant columns are present.")


In [ ]:
# a slight complication is that E-mail column might actually contain multiple emails separated by commas or /
# our goal is to convert this dataframe into a list of dictionaries with following keys:
# 'emails' (list of emails (str)), 'name' (str), 'surname' (str), 'api_key' (str)
# so let's write a function that takes a row and returns a dictionary
import re
def row_to_dict(row):
    emails = emails = [e for e in re.split(r'[,\s/]+', row["E-mail"]) if e]
    return {
        'emails': emails,
        'name': row['Name'],
        'surname': row['Surname'],
        'api_key': row['api_key']
    }



In [ ]:
# now let's create that list of dictionaries
participants = df.apply(row_to_dict, axis=1).tolist()
print(f"Converted {len(participants)} participants to list of dictionaries.")

In [ ]:
# Count participants with more than one email address without printing personal data.
multiple_email_count = sum(1 for participant in participants if len(participant["emails"]) > 1)
print(f"Participants with multiple email addresses: {multiple_email_count}")


In [ ]:
import time

default_bulk_provider = "SENDGRID" if my_sendgrid_key else "SMTP2GO"
bulk_provider = _normalise_provider(os.getenv("EMAIL_PROVIDER", default_bulk_provider))
bulk_key_by_provider = {
    "SENDGRID": my_sendgrid_key,
    "SMTP2GO": my_smtp2go_key,
}
bulk_api_key = bulk_key_by_provider[bulk_provider]
if not bulk_api_key:
    raise ValueError(f"The API key for {bulk_provider} is not set in the environment")

BULK_EMAIL_SUBJECT = "Your Unique API Key for the BSSDH Workshop on August 7th"
DELAY = 0.2  # delay in seconds to avoid hitting provider rate limits
sent_count = 0
failed_count = 0

print(f"Sending emails with {bulk_provider} and a delay of {DELAY} seconds to avoid rate limits.")
for participant_index, participant in enumerate(participants, start=1):
    for email_index, email in enumerate(participant["emails"], start=1):
        message = (
            f"Dear {participant['name']},\n\n"
            "Thank you for participating in our Using LLMs in Humanities Research via API workshop.\n\n"
            f"Your unique API key is:\n{participant['api_key']}\n\n"
            "Please keep this key secure and do not share it with others.\n\n"
            "This key is essential for actively participating in the workshop.\n"
            "The official repository for this workshop is available at:\n"
            "https://github.com/ValRCS/BSSDH_2025_workshop_LLM_API\n"
            "We will provide instructions at the workshop on how to access the repository and use the provided materials.\n\n"
            "See you on August 7th!\n\n"
            "According to the workshop schedule on: https://www.digitalhumanities.lv/bssdh/2025/Programme/\n"
            "The first session of this particular workshop will start at 11:30AM\n"
            "There is another workshop before ours which does not require this key.\n\n"
            "There is no need to reply to this e-mail as everything will be explained at the workshop.\n\n"
            "Best regards,\n"
            "On behalf of the BSSDH Workshop Team - Valdis Saulespurens\n"
        )
        try:
            result = _send_email(
                provider=bulk_provider,
                key=bulk_api_key,
                to_email=email,
                subject=BULK_EMAIL_SUBJECT,
                message=message,
            )
            sent_count += 1
            print(
                f"Sent participant {participant_index}, address {email_index}. Status: {result['status_code']}"
            )
        except Exception as e:
            failed_count += 1
            print(f"Failed participant {participant_index}, address {email_index}: {type(e).__name__}")
        time.sleep(DELAY)

print(f"All emails processed. Sent: {sent_count}; failed: {failed_count}.")
